# 04 — Forecasting Benchmark (FB-FC-v1)

**Purpose:** benchmark simple, reproducible forecasters of the future arrival rate (and gas demand), with a
chronological, leakage-safe protocol.

**How the benchmark was run:**
- The executed runs are `run_flashbots_benchmark.py select` (21 Sep only) and `final` (12 Oct, once, gated by the
  SHA-256 of the frozen selection).
- Stored outputs are in `Flashbots/results/FB-FC-v1/`.

**What this notebook does:**
1. explains the protocol and models, and proves causality;
2. **recomputes the whole selection stage in memory** and asserts it equals the frozen selection;
3. **re-checks the stored test-day scores**: it recomputes them with the frozen parameters and asserts equality.
   This is a reproduction of numbers already published, not a new test. No model is re-selected on 12 Oct.

**12 Oct was already exposed.** The pre-registered P2 benchmark already opened 12 Oct (arrival rate at H = 10 and
60 s) on 2026-11-14. FB-FC-v1 makes no choice using 12 Oct, but its 12 Oct numbers are a second look at that day.

This notebook writes no files.

## Setup

In [1]:
from pathlib import Path
import sys, json

ROOT = Path.cwd()
while not (ROOT / "Flashbots" / "fbcommon.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
FB = ROOT / "Flashbots"
for sub in ("", "forecasting", "evaluation", "time_series", "data_audit", "preprocessing"):
    sys.path.insert(0, str(FB / sub))

import numpy as np
import pandas as pd
import fbcommon as fc
fc.plot_style()
%matplotlib inline
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.0f}" if abs(v) >= 1e4 else f"{v:.4g}")
print("repository:", ROOT)
print({k: v for k, v in fc.environment().items() if k != "platform"})

repository: C:\OJT_SEM3\AI-Assisted-Permissioned-Blockchain-Architecture-


{'python': '3.14.3', 'numpy': '2.5.4', 'pandas': '3.0.6', 'scipy': '1.18.1', 'scikit_learn': '1.9.1', 'matplotlib': '3.11.2', 'pyyaml': '6.0.3'}


In [2]:
import models as fm
import metrics as em
from fabric_perf.forecasting import horizon_benchmark as hb

CONFIG = json.loads((FB / "experiments" / "configs" / "FB-FC-v1.json").read_text(encoding="utf-8"))
RES = FB / "results" / "FB-FC-v1"
FROZEN = json.loads((RES / "select" / "selection_frozen.json").read_text(encoding="utf-8"))
print("frozen selection SHA-256:", fc.sha256_file(RES / "select" / "selection_frozen.json"))

frozen selection SHA-256: 7823173D941B7B691A00D7FFE89FD16DCC38B57307CDEE2BF9A23859CA898AA6


## 1. Protocol: chronological pools, horizons, origins

In [3]:
pools = pd.DataFrame(CONFIG["pools"]).T
pools["first_origin_utc"] = pd.to_timedelta(pools["first_origin_s"].astype(int), unit="s").astype(str)
pools["end_utc"] = pd.to_timedelta(pools["end_s"].astype(int), unit="s").astype(str)
display(pools[["date", "first_origin_utc", "end_utc", "use"]])
pd.DataFrame({f"H = {h} s": {p: len(fm.origins(CONFIG["pools"][p]["first_origin_s"], CONFIG["pools"][p]["end_s"], h)) for p in "DVF"}
              for h in CONFIG["horizons_s"]}).rename_axis("non-overlapping origins")

,date,first_origin_utc,end_utc,use
D,2026-09-21,0 days 00:15:00,0 days 12:00:00,"fit: EWMA alpha, ridge coefficients and lambda..."
V,2026-09-21,0 days 12:00:00,1 days 00:00:00,model selection only
F,2026-10-12,0 days 00:15:00,1 days 00:00:00,scored once after the selection is frozen


,H = 1 s,H = 10 s,H = 60 s,H = 300 s
non-overlapping origins,,,,
D,42300,4230,705,141
V,43200,4320,720,144
F,85500,8550,1425,285


**Pools:**
- **D** (21 Sep 00:15-12:00) fits everything that is fitted: the EWMA alpha, the ridge coefficients and lambda
  (5-fold expanding time-series CV inside D), and the regime cut-offs.
- **V** (21 Sep 12:00-24:00) only **selects**: lowest MAE wins; models within 1% are tied and the simpler one wins.
- **F** (12 Oct) is scored once after the selection file is frozen.

There are no random splits, and the days are never joined.

**Horizons:**
- **1 s:** the Caliper replay resolution.
- **10 s:** below one Ethereum slot.
- **60 s:** the controller decision interval.
- **300 s:** a configuration-change timescale.

## 2. Models

In [4]:
pd.DataFrame([{"model": m, "definition": d} for m, d in CONFIG["models"].items()])

,model,definition
0,persistence_interval,naive / persistence: mean of the last H second...
1,seasonal_naive_12,"mean of [t-12, t-12+H): same phase one Ethereu..."
2,seasonal_naive_60,"mean of [t-60, t-60+H); only when H < 60"
3,seasonal_naive_300,"mean of [t-300, t-300+H); only when H < 300"
4,rolling_mean_60,mean of the last 60 s; omitted when 60 = H (id...
5,rolling_mean_300,mean of the last 300 s; omitted when 300 = H
6,rolling_mean_900,mean of the last 900 s
7,expanding_mean,mean of everything since 00:00 UTC of the same...
8,ewma,"exponentially weighted mean of the 1-s series,..."
9,ridge_ar,"ridge regression on trailing means over 1, 5, ..."


In [5]:
pd.DataFrame({f"H = {h} s": {m: m in fm.models_for_horizon(h) for m in fm.SELECTABLE_ORDER} for h in CONFIG["horizons_s"]}).rename_axis("model used at horizon")

,H = 1 s,H = 10 s,H = 60 s,H = 300 s
model used at horizon,,,,
persistence_interval,True,True,True,True
seasonal_naive_12,True,True,False,False
seasonal_naive_60,True,True,False,False
seasonal_naive_300,True,True,True,False
rolling_mean_60,True,True,False,True
rolling_mean_300,True,True,True,False
rolling_mean_900,True,True,True,True
expanding_mean,True,True,True,True
ewma,True,True,True,True


Variants that would read the future (seasonal period <= H) or that are identical to persistence (rolling window = H)
are dropped automatically. A daily (24 h) seasonal naive cannot exist inside a one-day series. Its cross-date
stand-in, the 21 Sep time-of-day profile, is exploratory and not selectable.

## 3. Causality check: changing the future must not change a forecast

In [6]:
day = fc.load_target_series("2026-09-21")["arrival_count"].to_numpy(float)
origin = np.array([50_000])
future_changed = day.copy(); future_changed[50_000:] = 1e6
rows = []
for H in CONFIG["horizons_s"]:
    sel = FROZEN["selection"][f"arrival_rate|{H}"]
    ridge = json.loads((ROOT / sel["ridge_params_file"]).read_text(encoding="utf-8"))
    for m in fm.models_for_horizon(H):
        a = fm.forecast(m, day, origin, H, sel["ewma_alpha"], ridge)[0]
        b = fm.forecast(m, future_changed, origin, H, sel["ewma_alpha"], ridge)[0]
        rows.append({"H": H, "model": m, "forecast": a, "forecast with future set to 1e6": b, "unchanged": a == b})
causal = pd.DataFrame(rows)
assert causal["unchanged"].all()
print("all", len(causal), "model x horizon forecasts are unchanged when every value from the origin onwards is replaced")

all 33 model x horizon forecasts are unchanged when every value from the origin onwards is replaced


## 4. Selection stage recomputed in memory (21 Sep only) and compared with the frozen selection

In [7]:
d0, d1 = CONFIG["pools"]["D"]["first_origin_s"], CONFIG["pools"]["D"]["end_s"]
v0, v1 = CONFIG["pools"]["V"]["first_origin_s"], CONFIG["pools"]["V"]["end_s"]
sep = fc.load_target_series("2026-09-21")
rows, val_tables = [], []
for target, spec in CONFIG["targets"].items():
    series = sep[spec["column"]].to_numpy(float)
    scale = float(spec.get("report_scale", 1))
    series_d = series[:d1]                      # D only: nothing after 12:00 UTC is visible to fitting
    for H in CONFIG["horizons_s"]:
        org_d, org_v = fm.origins(d0, d1, H), fm.origins(v0, v1, H)
        alpha, _ = fm.tune_ewma(series_d, org_d, H, CONFIG["ewma_alpha_grid"])
        ridge, _ = fm.fit_ridge(series_d, org_d, H, CONFIG["ridge"]["windows_s"], CONFIG["ridge"]["lambda_grid"], CONFIG["ridge"]["cv_folds"])
        y = fm.targets(series, org_v, H)
        val = {m: em.mae(y / scale, fm.forecast(m, series, org_v, H, alpha, ridge) / scale) for m in fm.models_for_horizon(H)}
        chosen = fm.select(val, CONFIG["near_tie"])
        frozen = FROZEN["selection"][f"{target}|{H}"]
        rows.append({"target": target, "H": H, "alpha": alpha, "ridge lambda": ridge["lambda"], "selected": chosen,
                     "validation MAE": val[chosen], "frozen selected": frozen["selected_model"],
                     "matches frozen": chosen == frozen["selected_model"] and alpha == frozen["ewma_alpha"]
                                       and ridge["lambda"] == frozen["ridge_lambda"]
                                       and all(np.isclose(val[m], frozen["validation_mae"][m], rtol=0, atol=1e-12) for m in val)})
        val_tables.append(pd.Series(val, name=(target, H)))
selection = pd.DataFrame(rows)
display(selection)
assert selection["matches frozen"].all()

,target,H,alpha,ridge lambda,selected,validation MAE,frozen selected,matches frozen
0,arrival_rate,1,0.1,100,ridge_ar,4.78,ridge_ar,True
1,arrival_rate,10,0.02,100,ridge_ar,2.323,ridge_ar,True
2,arrival_rate,60,0.005,10,ewma,1.32,ewma,True
3,arrival_rate,300,0.002,10,ewma,0.8247,ewma,True
4,gas_demand_rate,1,0.002,0.01,rolling_mean_60,1.962,rolling_mean_60,True
5,gas_demand_rate,10,0.005,10,ewma,0.7688,ewma,True
6,gas_demand_rate,60,0.005,10,rolling_mean_900,0.3641,rolling_mean_900,True
7,gas_demand_rate,300,0.002,10,ewma,0.2143,ewma,True


In [8]:
pd.concat(val_tables, axis=1).rename_axis("validation MAE (21 Sep 12-24Z)")

arrival_rate                    gas_demand_rate                     
                                        1     10    60     300             1      10     60     300
validation MAE (21 Sep 12-24Z)                                                                     
persistence_interval                  6.155 2.957 1.682 0.9997            2.49  1.056 0.4597 0.2364
seasonal_naive_12                      6.69 2.856   NaN    NaN           2.447  1.036    NaN    NaN
seasonal_naive_60                     6.474 2.973   NaN    NaN           2.454 0.9798    NaN    NaN
seasonal_naive_300                    6.542  2.94 1.784    NaN           2.403 0.9838 0.4808    NaN
rolling_mean_60                        4.94 2.437   NaN  1.469           1.962 0.7982    NaN 0.3597
rolling_mean_300                      4.931 2.395  1.38    NaN           1.952 0.7724 0.3724    NaN
rolling_mean_900                      4.928 2.427 1.354 0.8761           1.954 0.7751 0.3641 0.2297
expanding_mean                        5.126 3.067 2.446  2.251           2.023 0.8226 0.4237 0.2963
ewma                                  4.924 2.379  1.32 0.8247           1.952 0.7688 0.3624 0.2143
ridge_ar                               4.78 2.323 1.343 0.9442           1.954 0.7641 0.3643 0.2129

**Near-tie rule.** For gas at H = 60 s, EWMA had the lowest validation MAE (0.3624), but rolling mean 900 s (0.3641)
lies within 1% of it and is simpler, so it was selected. This rule was fixed in the config before the run.

## 5. Stored test-day scores (12 Oct) re-checked with the frozen parameters

In [9]:
oct12 = fc.load_target_series("2026-10-12")
stored = pd.read_csv(RES / "final" / "scores_F.csv").set_index(["target", "horizon_s", "model"])
f0, f1 = CONFIG["pools"]["F"]["first_origin_s"], CONFIG["pools"]["F"]["end_s"]
worst = 0.0
for key, sel in FROZEN["selection"].items():
    spec = CONFIG["targets"][sel["target"]]; scale = float(spec.get("report_scale", 1)); H = sel["horizon_s"]
    series = oct12[spec["column"]].to_numpy(float)
    ridge = json.loads((ROOT / sel["ridge_params_file"]).read_text(encoding="utf-8"))
    org = fm.origins(f0, f1, H); y = fm.targets(series, org, H)
    preds = {m: fm.forecast(m, series, org, H, sel["ewma_alpha"], ridge) for m in sel["models"]}
    preds["tod_profile_0921"] = fm.tod_profile(sep[spec["column"]].to_numpy(float), org, H)
    for m, p in preds.items():
        worst = max(worst, abs(em.mae(y / scale, p / scale) - stored.loc[(sel["target"], H, m), "mae"]))
print(f"max |recomputed - stored| test MAE over all {len(stored)} model x target x horizon rows: {worst:.2e}")
assert worst < 1e-9

max |recomputed - stored| test MAE over all 74 model x target x horizon rows: 8.88e-16


In [10]:
f = stored.reset_index()
mae = f.pivot_table(index="model", columns=["target", "horizon_s"], values="mae")
skill = f.pivot_table(index="model", columns=["target", "horizon_s"], values="skill_mae_vs_persistence") * 100
print("Test-day MAE (arrival: tx/s, gas: Mgas/s)"); display(mae)
print("Skill vs naive persistence (%), > 0 = better"); display(skill.round(1))

Test-day MAE (arrival: tx/s, gas: Mgas/s)


target               arrival_rate                    gas_demand_rate                  
horizon_s                     1     10    60     300             1     10    60    300
model                                                                                 
ewma                        4.238 2.228  1.22 0.9399           2.741 1.971 1.497 1.357
expanding_mean              4.255 2.386 1.649  1.449           2.632 1.971 1.777 1.643
persistence_interval        4.989 2.923 1.482   1.18           2.714 2.242 1.625 1.442
ridge_ar                    4.064 2.195  1.21 0.9981           2.923 1.956 1.458 1.318
rolling_mean_300            4.275 2.222 1.303    NaN           2.776 1.977 1.534   NaN
rolling_mean_60             4.271 2.249   NaN  1.244           2.811 2.073   NaN 1.792
rolling_mean_900            4.283 2.235 1.284 0.9979           2.757 1.993 1.615 1.406
seasonal_naive_12           5.894 2.944   NaN    NaN           3.285 2.302   NaN   NaN
seasonal_naive_300          5.371 2.741 1.762    NaN           3.195 2.269 1.851   NaN
seasonal_naive_60           5.255  2.55   NaN    NaN           3.033 2.101   NaN   NaN
tod_profile_0921            6.041 3.262 2.331  2.036           3.089 2.017  1.72 1.601

Skill vs naive persistence (%), > 0 = better


target               arrival_rate                   gas_demand_rate                 
horizon_s                     1     10    60    300             1    10    60    300
model                                                                               
ewma                         15.1  23.8  17.7  20.3              -1 12.1   7.9   5.9
expanding_mean               14.7  18.4 -11.2 -22.8               3 12.1  -9.4 -13.9
persistence_interval            0     0     0     0               0    0     0     0
ridge_ar                     18.5  24.9  18.4  15.4            -7.7 12.7  10.2   8.6
rolling_mean_300             14.3    24  12.1   NaN            -2.3 11.8   5.6   NaN
rolling_mean_60              14.4    23   NaN  -5.4            -3.6  7.5   NaN -24.3
rolling_mean_900             14.2  23.5  13.3  15.4            -1.6 11.1   0.6   2.5
seasonal_naive_12           -18.1  -0.7   NaN   NaN             -21 -2.7   NaN   NaN
seasonal_naive_300           -7.7   6.2 -18.8   NaN           -17.7 -1.2 -13.9   NaN
seasonal_naive_60            -5.3  12.8   NaN   NaN           -11.8  6.3   NaN   NaN
tod_profile_0921            -21.1 -11.6 -57.3 -72.6           -13.8   10  -5.9   -11

In [11]:
sel_rows = f[f["selected"]][["target", "horizon_s", "model", "mae", "rmse", "bias", "smape_pct", "wape_pct", "skill_mae_vs_persistence"]]
dm = pd.read_csv(RES / "final" / "dm_F.csv")
dm_ref = dm[dm["comparator"] == "persistence_interval"][["target", "horizon_s", "stat", "p_holm_vs_persistence_family"]]
sel_rows.merge(dm_ref, on=["target", "horizon_s"]).rename(columns={"p_holm_vs_persistence_family": "DM p (Holm) vs naive"})

,target,horizon_s,model,mae,rmse,bias,smape_pct,wape_pct,skill_mae_vs_persistence,stat,DM p (Holm) vs naive
0,arrival_rate,1,ridge_ar,4.064,7.766,0.02999,38.69,38.43,0.1854,-65.52,0
1,arrival_rate,10,ridge_ar,2.195,3.615,0.003351,20.08,20.76,0.2492,-32.3,3.034e-215
2,arrival_rate,60,ewma,1.22,1.834,-0.06857,11.33,11.54,0.1769,-9.3,3.068e-19
3,arrival_rate,300,ewma,0.9399,1.257,-0.05801,8.799,8.889,0.2031,-7.141,3.902e-11
4,gas_demand_rate,1,rolling_mean_60,2.811,68.95,-0.0002987,69.95,104.2,-0.03572,0.4445,1
5,gas_demand_rate,10,ewma,1.971,30.17,-0.002423,34.37,73.06,0.1208,-1.083,1
6,gas_demand_rate,60,rolling_mean_900,1.615,12.67,-0.001557,22.95,59.87,0.006,-0.03319,1
7,gas_demand_rate,300,ewma,1.357,6.669,0.01982,18.43,50.32,0.05881,-0.6007,1


## 6. Agreement with the frozen P2 benchmark

P2 (`experiments/results/flashbots_horizon_benchmark_v2/`) was pre-registered for the arrival rate at H = 60 / 10 s.
Its 12 Oct forecasts were recomputed and match the stored ones to 3.6e-15 (`Flashbots/results/p2_reproduction/`).
At H = 60 s both benchmarks select **EWMA, alpha = 0.005**, with the same validation MAE (1.3198). The test MAE is
1.2259 (P2) vs 1.2201 (FB-FC-v1); the only difference is that FB-FC-v1's origins start at 900 s instead of 600 s.

In [12]:
rep = json.loads((FB / "results" / "p2_reproduction" / "p2_reproduction.json").read_text(encoding="utf-8"))
p2 = pd.read_csv(FB / "results" / "p2_reproduction" / "p2_extended_metrics_F.csv")
print("P2 reproduced exactly:", rep["reproduced_exactly"], "| max abs diff:", rep["max_abs_diff_any_forecast_or_target"])
p2[p2["selected_by_P2"]][["horizon_s", "model", "mae", "rmse", "smape_pct", "wape_pct", "bias"]]

P2 reproduced exactly: True | max abs diff: 3.552713678800501e-15


,horizon_s,model,mae,rmse,smape_pct,wape_pct,bias
7,60,ewma,1.226,1.844,11.37,11.59,-0.07293
12,10,rolling_mean_300,2.237,3.66,20.48,21.14,0.002789


## 7. TimesFM 2.5: not run

`timesfm` and `torch` are not installed in `.venv`, and running TimesFM needs a large download (package plus model
weights) that requires the owner's approval. It remains an optional offline candidate, and it is **not** part of the
Fabric controller. Notebook 05 shows that, for the current Fabric configuration menu, even a perfect forecast would
not change the decision. That makes a stronger forecaster low priority.

## Summary
- **Arrival rate.** The selected model beats naive persistence by **18-25%** in MAE at every horizon, all
  significant after Holm correction.
- **Selected models.** EWMA at 60 / 300 s; ridge AR at 1 / 10 s, where it gains only about 0.03-0.17 tx/s over
  EWMA.
- **Gas demand.** No significant gain over naive at any horizon.

Next: `../evaluation/05_flashbots_evaluation_and_fabric.ipynb`.